# 04 grounding evaluation

**Owner:** Daniel. Original notebook cells: 79, 81, 82, 83.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

# 7. RAG Grounding Evaluation

## Purpose
The project plan sets the RAG success criterion at **>= 90% of evaluation answers supported by retrieved source material**.

Build a fixed evaluation set before tuning the RAG pipeline. Questions should come from the approved clinic/public knowledge corpus and cover scheduling, insurance procedures, optical-order workflows, and staff guidance.

## Required fields
- `question_id`
- `question`
- `expected_source`
- `retrieved_source`
- `retrieval_relevant`
- `answer_supported`
- `insufficient_evidence_expected`
- `assistant_abstained_when_needed`
- `notes`

**Pass definition:** `answer_supported == True` for at least 90% of evaluable questions, with appropriate abstention when the corpus does not contain enough evidence.

**Owner:** Daniel  
**Reviewer:** Denis


## Execute Full 20-Question Grounding Evaluation

In [ ]:
# Daniel: Section 7 — RAG Grounding Evaluation & Summary (Consolidated)

import re
import time
import json
import numpy as np
import pandas as pd

TEAM_VOLUME = "/Volumes/workspace/default/eye_style_team2"
RAG_DATA_DIR = f"{TEAM_VOLUME}/rag"

def summarize_rag_eval(df):
    if df.empty:
        return {
            "n_questions": 0,
            "grounding_rate": None,
            "meets_90pct_grounding_target": None,
        }

    grounding_rate = float(df["answer_supported"].astype(float).mean())
    return {
        "n_questions": len(df),
        "grounding_rate": grounding_rate,
        "meets_90pct_grounding_target": grounding_rate >= 0.90,
    }


def verify_answer_grounding(question, answer, retrieved_chunks, retries=2):
    """
    Independent groundedness judge: verifies whether every factual claim in
    `answer` is strictly supported by `retrieved_chunks` (no outside facts).
    Uses a higher token budget to accommodate reasoning tokens in gpt-oss-120b.
    """
    llm_client, model_name = _get_rag_llm_client()
    context_str = "\n\n".join(
        f"[{c['doc_id']} — {c['section']}]\n{c['text']}"
        for c in retrieved_chunks
    )
    judge_prompt = (
        "You are a strict RAG grounding auditor.\n"
        "Compare the Assistant Answer ONLY against the Retrieved Context.\n"
        "Respond with a single raw JSON object and nothing else:\n"
        '{"supported": true, "rationale": "one concise sentence"}\n'
        "Set supported to true if every factual claim in the answer is directly "
        "supported by the retrieved context and answers the question accurately; "
        "set supported to false if there is any unsupported claim, fabrication, or hallucination."
    )

    last_raw = ""
    for attempt in range(retries):
        resp = llm_client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": judge_prompt},
                {
                    "role": "user",
                    "content": (
                        f"Question: {question}\n\n"
                        f"Retrieved Context:\n{context_str}\n\n"
                        f"Assistant Answer:\n{answer}"
                    ),
                },
            ],
            temperature=0.0,
            max_tokens=1800,
        )
        raw = _extract_llm_text(resp.choices[0].message.content).strip()
        last_raw = raw
        if not raw:
            continue

        # Extract JSON object even if wrapped in fences or prose
        match = re.search(r"\{.*\}", raw, flags=re.DOTALL)
        if match:
            try:
                parsed = json.loads(match.group(0))
                if "supported" in parsed:
                    return {
                        "supported": bool(parsed["supported"]),
                        "rationale": str(parsed.get("rationale", "")).strip() or "Verified against retrieved context.",
                    }
            except json.JSONDecodeError:
                pass

    supported = bool(re.search(r'"supported"\s*:\s*true', last_raw, flags=re.IGNORECASE))
    return {
        "supported": supported,
        "rationale": last_raw[:160] if last_raw else "Judge returned empty output after retries.",
    }


eval_rows = []
detailed_audit = []

print(
    f"Running RAG grounding evaluation on {len(RAG_EVAL_QUESTIONS)} fixed questions "
    f"(config={SELECTED_RAG_CONFIG['config_id']}, top_k={SELECTED_RAG_CONFIG['top_k']})..."
)

for item in RAG_EVAL_QUESTIONS:
    qid = item["question_id"]
    started = time.perf_counter()

    res = ask_rag(
        question=item["question"],
        authenticated_role=item["role"],
        authenticated_subject=f"eval-{qid}",
        index=rag_index,
        top_k=SELECTED_RAG_CONFIG["top_k"],
    )
    elapsed = round(time.perf_counter() - started, 3)

    retrieved_doc_ids = list(dict.fromkeys(c["doc_id"] for c in res["retrieved_chunks"]))
    retrieved_source_str = ", ".join(retrieved_doc_ids) if retrieved_doc_ids else "NONE"
    abstained = bool(res["abstained"])
    expect_abstain = bool(item["insufficient_evidence_expected"])

    if expect_abstain:
        retrieval_relevant = False
        abstained_when_needed = abstained
        answer_supported = np.nan if abstained else False
        note = (
            f"Correctly abstained (top_sim={res['top_similarity']}, {elapsed}s)."
            if abstained
            else f"FAILED TO ABSTAIN on out-of-corpus query (top_sim={res['top_similarity']})."
        )
    else:
        retrieval_relevant = item["expected_source"] in retrieved_doc_ids
        abstained_when_needed = np.nan
        if abstained:
            answer_supported = False
            note = f"False abstention on in-corpus question (top_sim={res['top_similarity']})."
        else:
            verdict = verify_answer_grounding(
                item["question"], res["answer"], res["retrieved_chunks"]
            )
            answer_supported = bool(verdict["supported"] and retrieval_relevant)
            note = f"{verdict['rationale']} (top_sim={res['top_similarity']}, {elapsed}s)"

    eval_rows.append({
        "question_id": qid,
        "question": item["question"],
        "expected_source": item["expected_source"],
        "retrieved_source": retrieved_source_str,
        "retrieval_relevant": retrieval_relevant,
        "answer_supported": answer_supported,
        "insufficient_evidence_expected": expect_abstain,
        "assistant_abstained_when_needed": abstained_when_needed,
        "notes": note,
    })

    detailed_audit.append({
        "question_id": qid,
        "role": item["role"],
        "question": item["question"],
        "expected_source": item["expected_source"],
        "retrieved_sources": retrieved_doc_ids,
        "cited_sources": res["sources"],
        "top_similarity": res["top_similarity"],
        "abstained": abstained,
        "insufficient_evidence_expected": expect_abstain,
        "answer_supported": None if pd.isna(answer_supported) else bool(answer_supported),
        "seconds": elapsed,
        "answer": res["answer"],
        "notes": note,
    })

    status_tag = (
        "ABSTAIN_PASS" if (expect_abstain and abstained)
        else "ABSTAIN_FAIL" if expect_abstain
        else "GROUNDED_PASS" if answer_supported
        else "GROUNDED_FAIL"
    )
    print(f"  {qid}: {status_tag} | expected={item['expected_source']} | retrieved={retrieved_source_str} | {elapsed}s")

# Populate Section 7 rag_eval DataFrame with all 9 required fields
rag_eval = pd.DataFrame(eval_rows, columns=[
    "question_id",
    "question",
    "expected_source",
    "retrieved_source",
    "retrieval_relevant",
    "answer_supported",
    "insufficient_evidence_expected",
    "assistant_abstained_when_needed",
    "notes",
])

display(rag_eval)

section7_summary = summarize_rag_eval(rag_eval)

in_corpus_df = rag_eval[~rag_eval["insufficient_evidence_expected"]]
out_corpus_df = rag_eval[rag_eval["insufficient_evidence_expected"]]

rag_eval_full_summary = {
    **section7_summary,
    "n_evaluable_in_corpus": int(len(in_corpus_df)),
    "retrieval_hit_rate_in_corpus": round(float(in_corpus_df["retrieval_relevant"].mean()), 4),
    "in_corpus_grounding_rate": round(float(in_corpus_df["answer_supported"].astype(float).mean()), 4),
    "n_insufficient_evidence": int(len(out_corpus_df)),
    "abstention_accuracy": round(float(out_corpus_df["assistant_abstained_when_needed"].astype(float).mean()), 4),
    "overall_pass_rate": round(
        float(
            (
                int(in_corpus_df["answer_supported"].eq(True).sum())
                + int(out_corpus_df["assistant_abstained_when_needed"].eq(True).sum())
            )
            / len(rag_eval)
        ),
        4,
    ),
    "median_rag_seconds": round(float(np.median([r["seconds"] for r in detailed_audit])), 3),
    "selected_rag_config": SELECTED_RAG_CONFIG,
}

# Export results to Unity Catalog Volume for Week 6 Results & Denis's peer review
rag_eval.to_csv(f"{RAG_DATA_DIR}/rag_eval_results.csv", index=False)
with open(f"{RAG_DATA_DIR}/rag_eval_detailed_audit.json", "w", encoding="utf-8") as f:
    json.dump(detailed_audit, f, indent=2)
with open(f"{RAG_DATA_DIR}/rag_eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(rag_eval_full_summary, f, indent=2)

print("\n=== SECTION 7 OFFICIAL SUMMARY ===")
print(json.dumps(section7_summary, indent=2))

print("\n=== EXTENDED RAG EVALUATION BREAKDOWN ===")
print(json.dumps(rag_eval_full_summary, indent=2))

In [ ]:
# Inspect the 2 in-corpus questions that failed grounding under top_k=1
failed = rag_eval[(~rag_eval["insufficient_evidence_expected"]) & (rag_eval["answer_supported"] != True)]
for qid in failed["question_id"]:
    rec = next(r for r in detailed_audit if r["question_id"] == qid)
    print(f"=== {qid} | expected={rec['expected_source']} | retrieved={rec['retrieved_sources']} ===")
    print("Q:", rec["question"])
    print("A:", rec["answer"])
    print("Judge note:", rec["notes"], "\n")